In [4]:
# 06 - Hyperparameter Tuning (with preprocessing Pipeline)

import os
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
import joblib

# Load dataset
df = pd.read_csv("../data/heart_disease_cleaned.csv")
df["target"] = df["target"].apply(lambda x: 1 if x > 0 else 0)

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Columns (robust to already-encoded data)
categorical_candidates = ["cp", "restecg", "slope", "thal", "ca"]
categorical_cols = [c for c in categorical_candidates if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="drop",
)

# ===============================
# 1. Random Forest Tuning
# ===============================
rf = RandomForestClassifier(random_state=42)
pipeline_rf = Pipeline(steps=[("preprocessor", preprocessor),
                              ("classifier", rf)])

param_grid_rf = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5],
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring="f1", n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Best RF params:", grid_rf.best_params_)
print("Best RF CV score:", grid_rf.best_score_)

# ===============================
# 2. SVM Tuning
# ===============================
svm = SVC(probability=True, random_state=42)
pipeline_svm = Pipeline(steps=[("preprocessor", preprocessor),
                               ("classifier", svm)])

param_grid_svm = {
    "classifier__C": [0.1, 1, 10],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto"],
}

grid_svm = GridSearchCV(pipeline_svm, param_grid_svm, cv=5, scoring="f1", n_jobs=-1)
grid_svm.fit(X_train, y_train)

print("Best SVM params:", grid_svm.best_params_)
print("Best SVM CV score:", grid_svm.best_score_)

# Pick the best pipeline by CV score
best_grid = grid_rf if grid_rf.best_score_ >= grid_svm.best_score_ else grid_svm
best_name = "Random Forest" if best_grid is grid_rf else "SVM"
best_pipeline = best_grid.best_estimator_

print(f"\nSelected best model: {best_name}")
print("\nTest Report:\n", classification_report(y_test, best_pipeline.predict(X_test)))

# Save best pipeline
os.makedirs("../models", exist_ok=True)
joblib.dump(best_pipeline, "../models/final_model.pkl")
print("✅ Final pipeline saved to ../models/final_model.pkl")

Best RF params: {'classifier__max_depth': None, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Best RF CV score: 0.803877312855015
Best SVM params: {'classifier__C': 10, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
Best SVM CV score: 0.8214551642345176

Selected best model: SVM

Test Report:
               precision    recall  f1-score   support

           0       0.93      0.79      0.85        33
           1       0.79      0.93      0.85        28

    accuracy                           0.85        61
   macro avg       0.86      0.86      0.85        61
weighted avg       0.86      0.85      0.85        61

✅ Final pipeline saved to ../models/final_model.pkl
